# MQTT Integration - DJI Dock 3 + AWS IoT Core @ 3fps

**Author:** Ali Naderi | Edge AI  
**Purpose:** Show team how to configure drone/Dock 3 to output JSON over MQTT at ~3fps instead of video stream

---

## Architecture

```
Matrice 4TD NPU (YOLOv8n INT8, ~45ms)
    ↓
Pilot 2 throttles to 3fps JSON
    ↓ MQTT publish
DJI Cloud API / AWS IoT Core (topic: dji/matrice4td/inference)
    ↓
Your Backend (this notebook + subscriber script)
```

Why JSON not video? Bandwidth 1000x savings, privacy, battery.


## 1. Setup & Payload Models

Type-safe validation with Pydantic - production extras.

In [ ]:
import sys
from pathlib import Path
import os

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.mqtt.payload_models import Detection, InferencePayload, AlertPayload
import json

print("✅ Payload models imported")

# Load sample
with open("demo/sample_payload.json", 'r') as f:
    sample = json.load(f)

inference = InferencePayload(**sample)
print(f"Sample: {inference.to_summary()}")
print(f"Safety violations: {len(inference.get_safety_violations())}")


## 2. Local Testing with EMQX (No Drone, No AWS)

Simulates full pipeline for SOP validation.


In [ ]:
import socket

def check_broker(host="localhost", port=1883):
    try:
        s = socket.socket()
        s.settimeout(2)
        result = s.connect_ex((host, port))
        s.close()
        return result == 0
    except:
        return False

broker_ok = check_broker()
print(f"EMQX broker at localhost:1883 running: {broker_ok}")

if not broker_ok:
    print("\nTo start broker:")
    print("  docker-compose -f docker/docker-compose.yml up -d")
    print("  Dashboard: http://localhost:18083 admin/public")
else:
    print("✅ Broker running")


In [ ]:
# If broker running, demonstrate mock publisher logic (without blocking)
if broker_ok:
    from src.mqtt.dji_mock_publisher import DJIMockPublisher
    
    # Create publisher instance (don't start blocking loop in notebook)
    publisher = DJIMockPublisher(endpoint="localhost", port=1883, topic="dji/matrice4td/inference", fps=3.0)
    
    # Generate 3 sample payloads
    print("Generating 3 sample payloads @ 3fps (mock):\n")
    for i in range(3):
        payload = publisher._generate_payload()
        print(f"Frame {payload['frame_id']}: {len(payload['detections'])} detections")
        for det in payload['detections'][:2]:
            print(f"  - {det['class_name']} {det['confidence']} {det['bbox']}")
        print()
else:
    print("Skipping live mock (broker not running) - payload models already validated")


## 3. AWS IoT Core Setup (Production)

Detailed steps in `docs/MQTT_SETUP.md`. Summary:

1. Create Thing `matrice-4td-01`
2. Create certs → save to `./certs/`
3. Create Policy allowing `iot:Publish, Subscribe, Connect` on `dji/*`
4. Attach policy to cert, cert to Thing
5. Get endpoint `xxxxxx-ats.iot.us-east-1.amazonaws.com`
6. Configure Dock 3 Cloud API to use AWS endpoint + certs, topic `dji/matrice4td/inference`, JSON @ 3fps


In [ ]:
# Check if certs exist
certs_dir = Path("./certs")
if certs_dir.exists():
    files = list(certs_dir.glob("*"))
    print(f"Certs dir exists: {certs_dir}, files: {[f.name for f in files]}")
    
    required = ["AmazonRootCA1.pem", "certificate.pem.crt", "private.pem.key"]
    for req in required:
        exists = (certs_dir / req).exists()
        print(f"  {req}: {'✅' if exists else '❌ missing'}")
else:
    print(f"Certs dir not found: {certs_dir}")
    print("For AWS IoT, create certs via IoT Core console and save to ./certs/")
    print("For local test, no certs needed")


## 4. Subscriber - Receive JSON @ 3fps

This is what your backend team will run to receive inference results.


In [ ]:
from src.mqtt.aws_subscriber import DJIInferenceSubscriber

# Example: Local subscriber (no TLS)
print("Example subscriber setup (local):")
print("""
subscriber = DJIInferenceSubscriber(
    endpoint="localhost",
    port=1883,
    topic="dji/matrice4td/inference",
    use_tls=False,
    log_file="demo/dji_inference_log.jsonl"
)
subscriber.start()  # Blocking, prints detections @ 3fps
""")

print("\nExample subscriber setup (AWS IoT):")
print("""
subscriber = DJIInferenceSubscriber(
    endpoint="xxxxxx-ats.iot.us-east-1.amazonaws.com",
    port=8883,
    topic="dji/matrice4td/inference",
    use_tls=True,
    cert_dir="./certs",
    log_file="demo/dji_inference_log.jsonl"
)
subscriber.start()
""")

print("\nFor live test in terminal (not notebook, because blocking):")
print("  python scripts/run_mqtt_test.py --mode subscriber --endpoint localhost")


## 5. Safety Alert Logic (Over-Delivery)

Extra: automatic alert when No-Hard-Hat detected.

In [ ]:
# Simulate alert handling
from src.mqtt.payload_models import InferencePayload, AlertPayload
import time

# Create a violation payload
violation_payload = {
    "timestamp": int(time.time()*1000),
    "drone_sn": "4TD-TEST-001",
    "frame_id": 999,
    "detections": [
        {"class_id": 0, "class_name": "Person", "confidence": 0.92, "bbox": [100, 200, 150, 300]},
        {"class_id": 3, "class_name": "No-Hard-Hat", "confidence": 0.88, "bbox": [105, 205, 145, 235]}
    ],
    "inference_time_ms": 45.2,
    "model_version": "yolov8n_dji_4class_v1"
}

inference = InferencePayload(**violation_payload)
print(f"Inference: {inference.to_summary()}")

alert = AlertPayload.from_inference(inference)
if alert:
    print(f"\n⚠️ ALERT GENERATED:")
    print(f"  Type: {alert.alert_type}")
    print(f"  Severity: {alert.severity}")
    print(f"  Message: {alert.message}")
    print(f"  Drone: {alert.drone_sn}")
    print(f"  Violations: {len(alert.violations)}")
    
    # In production, you would:
    # - Publish to alerts/safety topic
    # - Send to SNS
    # - Save to DynamoDB
    # - Trigger Lambda
    print(f"\nIn production, this would trigger SNS, Lambda, or republish to alerts/safety")


## 6. Validation for Deliverable

To prove end-to-end that team can receive MQTT JSON in AWS broker:

1. Run subscriber for 5 minutes
2. Should receive ~900 messages (3fps * 300s)
3. Log file `demo/dji_inference_log.jsonl` with all 4 classes
4. Screenshot AWS IoT MQTT test client
5. Include in SOP deliverable


In [ ]:
# Check if log file exists from previous runs
log_path = Path("demo/dji_inference_log.jsonl")
if log_path.exists():
    with open(log_path, 'r') as f:
        lines = f.readlines()
    print(f"Log file exists: {log_path}")
    print(f"Total messages logged: {len(lines)}")
    
    if lines:
        import json
        last = json.loads(lines[-1])
        print(f"\nLast message sample:")
        print(json.dumps(last, indent=2)[:500] + "...")
else:
    print(f"Log file not yet created: {log_path}")
    print("Run subscriber + mock publisher to generate log")
    print("\nCommands:")
    print("  Terminal 1: python scripts/run_mqtt_test.py --mode subscriber")
    print("  Terminal 2: python scripts/run_mqtt_test.py --mode mock --frames 100")
    print("  After 5 mins, log will have ~900 entries")


## Summary

✅ Payload models with Pydantic validation  
✅ Local EMQX testing (docker-compose)  
✅ AWS IoT Core setup guide  
✅ Mock publisher @ 3fps for testing without drone  
✅ Safety alert logic (over-delivery)  
✅ Validation log for deliverable  

Next: See `docs/MQTT_SETUP.md` for full AWS setup and `docs/SOP.md` for end-to-end.
